# ⚠️ REQUIRED BEFORE RUNNING — Enable Free GPU

**If you skip this step, the notebook will crash or take 30+ minutes.**

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU**
4. Click **Save**

Then: **Runtime → Run all** (Ctrl+F9 / Cmd+F9)

---

# TunedAI Labs — Raw Passage Causal Reasoning Test

This notebook tests four AI models on **verbatim passages** from books published before computers existed.
Every passage below was retrieved directly from Project Gutenberg. No AI wrote these sentences.
The correct answers were established by historians, philosophers, and scientists — decades or centuries ago.

| Model | What it is |
|---|---|
| Base Qwen 2.5-7B | An unmodified open-source model |
| GPT-4o | OpenAI's best model (optional — needs API key) |
| Claude 3.5 Sonnet | Anthropic's best model (optional — needs API key) |
| **TunedAI Labs Causal Model** | The same Qwen 2.5-7B, fine-tuned by TunedAI Labs on causal reasoning |

---

### Sources used

| Author | Work | Year | Retrieved from |
|---|---|---|---|
| David Hume | An Enquiry Concerning Human Understanding | 1748 | Project Gutenberg #9662 |
| John Snow | On the Mode of Communication of Cholera (2nd ed.) | 1855 | Project Gutenberg #72894 |
| John Stuart Mill | A System of Logic | 1843 | Project Gutenberg #27942 |
| Florence Nightingale | Notes on Nursing | 1860 | Project Gutenberg #17366 |

These texts were written 85–278 years before the first AI language model. The sentence structures are provably pre-AI.
Our full benchmark (96.96% on 10,112 questions) uses the same reasoning applied across a much wider range of problems.


---
## Cell 1 of 4 — Install required packages

This installs the software needed to run the models. You will see a lot of text scroll by — that is normal. Wait for it to finish before the next cell runs.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes torch openai anthropic

---
## Cell 2 of 4 — Optional: Add your API keys

If you want to include **GPT-4o** and **Claude 3.5** in the comparison, paste your API keys below between the quotes.

- OpenAI key: get one at **platform.openai.com** → API Keys
- Anthropic key: get one at **console.anthropic.com** → API Keys

**If you leave these blank, those columns will be skipped.** Base Qwen vs TunedAI Labs still runs with no keys needed.

In [ ]:
OPENAI_API_KEY    = ""   # paste your OpenAI key here (optional)
ANTHROPIC_API_KEY = ""   # paste your Anthropic key here (optional)

---
## Cell 3 of 4 — Load the models

This downloads and loads two versions of the same AI model:
- The original unmodified version (Base Qwen 2.5-7B)
- TunedAI Labs fine-tuned version with causal reasoning training

**This takes about 2 minutes.** You will see messages like "Loading tokenizer", "Loading base model", "Loading TunedAI Labs adapter". Wait for the green checkmark ✓ before continuing.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import openai, anthropic

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedailabs/causal-reasoning-qwen-7b"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("Loading base model (largest step — about 90 seconds)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Loading TunedAI Labs causal reasoning adapter...")
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("\n✓ All models ready. Scroll down to see the results.")

---
## Cell 4 of 4 — Helper functions

This sets up the comparison engine. It runs automatically — nothing to change here.

In [ ]:
SYSTEM = "You are a careful reasoner. Answer questions about causation, association, intervention, and counterfactuals precisely and correctly."

def ask_local(question, use_adapter=True, max_new_tokens=350):
    if use_adapter:
        tuned_model.enable_adapter_layers()
    else:
        tuned_model.disable_adapter_layers()
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(tuned_model.device)
    with torch.no_grad():
        out = tuned_model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_gpt4(question):
    if not oai_client:
        return "[Skipped — no OpenAI API key provided]"
    r = oai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":question}],
        max_tokens=400
    )
    return r.choices[0].message.content.strip()

def ask_claude(question):
    if not ant_client:
        return "[Skipped — no Anthropic API key provided]"
    r = ant_client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=400,
        system=SYSTEM,
        messages=[{"role":"user","content":question}]
    )
    return r.content[0].text.strip()

def compare_all(question, source="", note=""):
    SEP = "=" * 70
    DIV = "-" * 70
    print(SEP)
    if source: print(f"SOURCE : {source}")
    if note:   print(f"NOTE   : {note}")
    print(SEP)
    print(f"QUESTION:\n{question}\n")
    for label, fn in [
        ("BASE QWEN 2.5-7B (untuned — the starting point)", lambda q: ask_local(q, use_adapter=False)),
        ("GPT-4o",                                          ask_gpt4),
        ("CLAUDE 3.5 SONNET",                               ask_claude),
        ("TUNEDAI CAUSAL MODEL ★",                          lambda q: ask_local(q, use_adapter=True)),
    ]:
        print(DIV)
        print(f"[ {label} ]")
        print(DIV)
        print(fn(question))
        print()

print("✓ Ready — running all tests now...")

---
# Results

Each test below shows a **verbatim passage** from a book written before computers existed, followed by a causal reasoning question about that passage.
The correct answers were established by historians, philosophers, and scientists — not by any AI system.

**BASE QWEN** = unmodified open-source model. **TUNEDAI LABS** = the same model after fine-tuning.

---

## Test 1 — Hume on Billiard Balls and Causal Connexion
**Source:** David Hume, *An Enquiry Concerning Human Understanding*, Section VII, Part II, §59 (1748)
**Retrieved from:** Project Gutenberg ebook #9662

The passage below is verbatim from the 1748 original.

In [ ]:
compare_all(
    question="""VERBATIM PASSAGE (Hume, 1748):

"The first time a man saw the communication of motion by impulse, as by the shock of two billiard balls, he could not pronounce that the one event was connected: but only that it was conjoined with the other. After he has observed several instances of this nature, he then pronounces them to be connected. What alteration has happened to give rise to this new idea of connexion? Nothing but that he now feels these events to be connected in his imagination, and can readily foretell the existence of one from the appearance of the other. When we say, therefore, that one object is connected with another, we mean only that they have acquired a connexion in our thought, and give rise to this inference, by which they become proofs of each other's existence."

QUESTION: According to this passage, what can a man conclude the FIRST time he sees one billiard ball strike another? After observing several such instances, Hume says he pronounces them "connected." What does Hume say has actually changed — and what has NOT changed in the physical world?""",
    source="David Hume, An Enquiry Concerning Human Understanding, Section VII Part II §59, 1748",
    note="Correct answer: The first time, the man can only say the events were conjoined — not connected. After repetition, a feeling of connexion forms in his imagination. What changed is purely mental: habit and custom. Nothing changed in the physical world — Hume says the 'connexion' exists in the mind, not in the objects themselves."
)


---
## Test 2 — Hume's Counterfactual Definition of Cause
**Source:** David Hume, *An Enquiry Concerning Human Understanding*, Section VII, Part II, §60 (1748)
**Retrieved from:** Project Gutenberg ebook #9662

In [ ]:
compare_all(
    question="""VERBATIM PASSAGE (Hume, 1748):

"Similar objects are always conjoined with similar. Of this we have experience. Suitably to this experience, therefore, we may define a cause to be an object, followed by another, and where all the objects similar to the first are followed by objects similar to the second. Or in other words where, if the first object had not been, the second never had existed."

QUESTION: Hume gives two definitions of a cause in this passage. State both definitions in your own words. According to his second definition, what question would you ask to test whether event A caused event B?""",
    source="David Hume, An Enquiry Concerning Human Understanding, Section VII Part II §60, 1748",
    note="Correct answer: First definition (regularity): a cause is an object followed by another where similar objects are always followed by similar objects — constant conjunction. Second definition (counterfactual): if the first object had not existed, the second would never have existed. To test whether A caused B: ask 'If A had not occurred, would B still have occurred?' If no, A caused B."
)


---
## Test 3 — John Snow's Natural Experiment
**Source:** John Snow, *On the Mode of Communication of Cholera*, 2nd edition, Chapter IX (1855)
**Retrieved from:** Project Gutenberg ebook #72894

In [ ]:
compare_all(
    question="""VERBATIM PASSAGE (Snow, 1855):

"As there is no difference whatever, either in the houses or the people receiving the supply of the two Water Companies, or in any of the physical conditions with which they are surrounded, it is obvious that no experiment could have been devised which would more thoroughly test the effect of water supply on the progress of cholera than this, which circumstances placed ready made before the observer. The experiment, too, was on the grandest scale. No fewer than three hundred thousand people of both sexes, of every age and occupation, and of every rank and station, from gentlefolks down to the very poor, were divided into two groups without their choice, and, in most cases, without their knowledge; one group being supplied with water containing the sewage of London, and, amongst it, whatever might have come from the cholera patients, the other group having water quite free from such impurity."

QUESTION: Snow claims this natural situation is equivalent to a planned experiment. What specific features of this situation does Snow say eliminate confounding? Why does Snow emphasize that the division into two groups occurred "without their choice, and, in most cases, without their knowledge"?""",
    source="John Snow, On the Mode of Communication of Cholera, 2nd edition, Chapter IX, 1855",
    note="Correct answer: The features that eliminate confounding: same houses, same people, same physical conditions in both groups — all factors except water supply are identical. Snow emphasizes that people did not choose their water supply and did not know which company served them, which rules out self-selection bias. This is functionally equivalent to random assignment in a controlled experiment."
)


---
## Test 4 — Snow's Intervention: Removing the Pump Handle
**Source:** John Snow, *On the Mode of Communication of Cholera*, 2nd edition (1855)
**Retrieved from:** Project Gutenberg ebook #72894

In [ ]:
compare_all(
    question="""VERBATIM PASSAGE (Snow, 1855):

"As soon as I became acquainted with the situation and extent of this irruption of cholera, I suspected some contamination of the water of the much-frequented street pump in Broad Street, near the end of Cambridge Street; but on examining the water, on the evening of the 3rd September, I found so little impurity in it of an organic nature, that I hesitated to come to a conclusion. Further inquiry, however, showed me that there was no other circumstance or agent common to the circumscribed locality in which this sudden increase of cholera occurred, and not extending beyond it, except the water of the above mentioned pump."

QUESTION: Snow says he "hesitated to come to a conclusion" after examining the water. What evidence did he use instead to overcome that hesitation? What logical method is Snow applying when he says there was "no other circumstance or agent common" to the affected area except the pump water?""",
    source="John Snow, On the Mode of Communication of Cholera, 2nd edition, 1855",
    note="Correct answer: Snow overcame his hesitation by using elimination — ruling out every other possible common factor in the affected area. The logical method is Mill's Method of Agreement (and elimination): the pump water was the only circumstance shared by all the cases and absent in the unaffected areas. The lack of visible impurity in the water was not sufficient to rule out the pump as a cause — the epidemiological pattern was stronger evidence than the chemical test."
)


---
## Test 5 — Mill's Method of Difference
**Source:** John Stuart Mill, *A System of Logic*, Book III, Chapter 8, Second Canon (1843)
**Retrieved from:** Project Gutenberg ebook #27942

In [ ]:
compare_all(
    question="""VERBATIM PASSAGE (Mill, 1843):

"If an instance in which the phenomenon under investigation occurs, and an instance in which it does not occur, have every circumstance in common save one, that one occurring only in the former; the circumstance in which alone the two instances differ, is the effect, or the cause, or an indispensable part of the cause, of the phenomenon."

And from Mill's example immediately following:

"It is scarcely necessary to give examples of a logical process to which we owe almost all the inductive conclusions we draw in daily life. When a man is shot through the heart, it is by this method we know that it was the gunshot which killed him: for he was in the fullness of life immediately before, all circumstances being the same, except the wound."

QUESTION: In the gunshot example, identify: (1) the two instances being compared, (2) what they have in common, (3) the one circumstance in which they differ, and (4) what Mill's method concludes from this. Why does Mill say this method is responsible for "almost all the inductive conclusions we draw in daily life"?""",
    source="John Stuart Mill, A System of Logic, Book III Chapter 8, Second Canon, 1843",
    note="Correct answer: (1) Two instances: the man immediately before the shot (alive) vs. the man after the shot (dead). (2) In common: all circumstances — his physical state, environment, everything. (3) The one difference: the gunshot wound. (4) Mill's conclusion: the wound is the cause of death. Mill says this method underlies most everyday causal inference because we constantly compare a situation before and after a single change to identify what caused a difference."
)


---
## Test 6 — Nightingale on Confounding in Hospital Statistics
**Source:** Florence Nightingale, *Notes on Nursing: What It Is, and What It Is Not*, Chapter on Cleanliness (1860)
**Retrieved from:** Project Gutenberg ebook #17366

In [ ]:
compare_all(
    question="""VERBATIM PASSAGE (Nightingale, 1860):

"In comparing the deaths of one hospital with those of another, any statistics are justly considered absolutely valueless which do not give the ages, the sexes, and the diseases of all the cases. It does not seem necessary to mention this. It does not seem necessary to say that there can be no comparison between old men with dropsies and young women with consumptions. Yet the cleverest men and the cleverest women are often heard making such comparisons, ignoring entirely sex, age, disease, place--in fact, all the conditions essential to the question. It is the merest gossip."

QUESTION: Nightingale calls unadjusted hospital death-rate comparisons "the merest gossip" and "absolutely valueless." What is the statistical problem she is identifying? What variables does she say must be accounted for, and why would ignoring them lead to a wrong conclusion about which hospital is better?""",
    source="Florence Nightingale, Notes on Nursing: What It Is, and What It Is Not, 1860",
    note="Correct answer: Nightingale is identifying confounding — hospitals treat different types of patients, so a higher death rate may reflect a sicker patient population, not worse care. Variables required: age, sex, disease type, and place. Without adjusting for these, you are comparing patient mix, not hospital quality. Comparing 'old men with dropsies' to 'young women with consumptions' is invalid because their baseline mortality is completely different — the hospital treating sicker patients will always appear worse even if it provides superior care."
)


---
## Try Your Own Question

Replace the text below with any causal reasoning question. Examples to try:
- "Does wearing a seatbelt cause people to drive more recklessly?"
- "If a rooster crows before sunrise every day, does the rooster cause the sun to rise?"
- "A city adds more police and crime goes up. Does adding police cause more crime?"

In [ ]:
MY_QUESTION = """
Type your question here and run this cell.
"""

compare_all(MY_QUESTION, source="Custom")

---
## Share What You Saw

Open a [GitHub Issue](https://github.com/tunedailabs/tunedailabs/issues/new) and paste your results.

Tell us which questions the TunedAI Labs model got right that the others did not — or anything surprising you found.

We read every submission.

---

**TunedAI Labs** — We fine-tune open-source LLMs for real-world reasoning.

[tunedailabs.com](https://tunedailabs.com)